# RMUC 2026 哨兵行为树插件文档

本文档介绍 `rmuc_2026` 行为树系统中所有自定义插件的功能、端口和使用方法。  
行为树主入口：`rmuc_2026.xml`，共引用 **13** 棵子树，涉及 **36** 个自定义插件。

> ⚠️ 消息类型已从单一 `RMUC.msg` 拆分为 **8 个独立小消息**，各自拥有独立话题。  
> 详见 `RMUC_Msg_Split_Doc.ipynb`。

---

## 1. 架构总览

```
rmuc_2026 (主树)
  ├── PerceptionAndBlackboard   → 5 话题订阅 + 黑板解析
  ├── InitOnce                  → 配置初始化
  ├── WhileDoElse (比赛阶段判断)
  │   ├── CommandHub            → 指令决策 + 裁判发送 (5Hz)
  │   └── ReactiveFallback (优先级从高到低)
  │       ├── [0] RespawnRecovery   → 死亡/复活/刷卡回血
  │       ├── [1] CriticalSurvival  → 危急生存
  │       ├── [2] BaseDefense       → 基地防御
  │       ├── [3] EngageCombat      → 交战
  │       ├── [4] SustainAndEconomy → 后勤补给
  │       ├── [5] ObjectivePlanner  → 目标占领
  │       └── [6] PatrolAndScan     → 巡逻扫描
  └── 非比赛阶段 → Home + 停火
```

## 2. 插件分类

| 分类 | 数量 | 消息类型 | 说明 |
|:---|:---:|:---|:---|
| **RMUC 镜像** (Rmuc 前缀) | 14 | 8 个拆分后的 `RMUC*.msg` | 从 RMUL 镜像，注册名带 `Rmuc` 前缀避免冲突 |
| **RMUC 特有新建** | 5 | 同上 | 全新设计，依赖 RMUC 拆分消息字段 |
| **通用新建** | 15+8=23 | 无 msg 依赖 | 仅使用黑板原始类型 |
| **共享复用** (rmul) | 8 | 无 | CancelNavGoal/SendGoal/MoveAround 等 |

## 3. RMUC 镜像插件 (14个)

这些插件与 RMUL 版本功能相同，但使用拆分后的 RMUC 消息类型（而非单一 `RMUC.msg`）。  
BT 注册名均带 `Rmuc` 前缀以避免运行时冲突。

### 3.1 订阅类动作节点

| BT 节点名 | 消息类型 | 话题 | 输出端口 |
|:---|:---|:---|:---|
| `RmucSubGameStatus` | `RMUCGameStatus` | `/game_status` | `game_status`, `now_ms` |
| `RmucSubRobotStatus` | `RMUCRobotStatus` | `/robot_status` | `robot_status` (shared_ptr) |
| `RmucSubRFIDStatus` | `RMUCRFIDStatus` | `/rfid_status` | `rfid_status` |
| `RmucSubRobotPosition` | `RMUCRobotPosition` | `/robot_position` | `pose_x`, `pose_y`, `pose_yaw`, `is_at_nav_goal` |

### 3.2 控制/导航动作节点

| BT 节点名 | 消息类型 | 话题 | 输入端口 |
|:---|:---|:---|:---|
| `RmucRobotControl` | `RMUCRobotControl` | `/robot_control` | `stop_gimbal_scan`, `chassis_spin`, `fire_enable` |
| `RmucNavControlCmd` | `RMUCNavControlCmd` | `/nav_control_cmd` | `cmd_type`, `emergency_stop` |
| `RmucWaitAndHeal` | `RMUCRobotStatus` | `/robot_status` | `now_ms`, `heal_start_ms`, `heal_wait_ms`, `hp_cur`, `hp_max`, `heal_min_ratio` |
| `RmucDetectRespawnAndSetRecovery` | `RMUCRobotStatus` | `/robot_status` | `was_dead`, `hp_cur`, `need_recovery`, ... |
| `RmucMicroSearchSupplyCard` | `RMUCRFIDStatus` | `/rfid_status` | `search_start_ms`, `timeout_ms`, ... |

### 3.3 条件节点

| BT 节点名 | 消息类型 | 输入端口 |
|:---|:---|:---|
| `RmucIsDead` | `RMUCRobotStatus` | `message` (shared_ptr) |
| `RmucIsGameTime` | `RMUCGameStatus` | `message`, `game_progress`, `lower/higher_remain_time` |
| `RmucIsHPBelow` | `RMUCRobotStatus` | `message` (shared_ptr), `hp_threshold` |
| `RmucIsSupplyCardDetected` | — | `rfid_status` (黑板 bool 读取) |
| `RmucIsAtNavGoal` | — | `is_at_nav_goal` (黑板 bool 读取) |

## 4. RMUC 特有新建插件 (5个)

这些插件是为 RMUC 2026 规则全新设计的，读写拆分后的 RMUC 消息类型。

### 4.1 SubRadarTracks
- **BT 注册名**: `SubRadarTracks`  
- **类型**: RosTopicSubNode (Action)  
- **消息**: `RMUCEnemyTracks` · 话题 `/radar/enemy_tracks`  
- **端口**: `topic_name` (in) → `radar_tracks` (out)

### 4.2 ParseSentryBlackboard
- **BT 注册名**: `ParseSentryBlackboard`  
- **类型**: SyncActionNode  
- **功能**: 从黑板读取 4 种原始消息（game/robot/radar + pose），解析派生 ~20 个状态变量  
- **关键输入**: `game_status`(`RMUCGameStatus`), `robot_status`(`RMUCRobotStatus`), `radar_tracks`(`RMUCEnemyTracks`), `pose_x/y`, `now_ms`  
- **关键输出**: `stage_remain_time`, `hp_cur/max`, `heat_cur`, `ammo_allow/left`, `base_hp_cur/max`, `outpost_alive`, `is_dead`, `is_weak`, `has_target`, `best_target`, `base_threat`, `team_coins`  
- **注意**: `rfid_status` 不再作为本节点输入（RFID 由 `IsZoneCardDetected` 等节点直接从黑板读取）

### 4.3 DecideRespawnCmd
- **BT 注册名**: `DecideRespawnCmd`  
- **类型**: SyncActionNode  
- **功能**: 决定复活确认/立即复活兑换  
- **输入**: `is_dead`, `robot_status`(`RMUCRobotStatus`), `team_coins`, `stage_remain_time`, `base_hp_cur/max`, `base_threat`  
- **输出**: `confirm_respawn`, `confirm_instant_respawn`  
- **策略**: 死亡时始终确认普通复活；基地受威胁且金币≥300 或剩余<60s 时兑换立即复活

### 4.4 SentryCmdMux
- **BT 注册名**: `SentryCmdMux`  
- **类型**: RosTopicPubNode (Action)  
- **消息**: `RMUCSentryCmd` · 话题 `/sentry_cmd`  
- **输入**: `posture`, `confirm_respawn`, `confirm_instant_respawn`, `allow_ammo_target`, `trigger_remote_ammo`, `trigger_remote_hp`, `enable_big_energy`, `cmd_state` (inout)

### 4.5 IsZoneCardDetected
- **BT 注册名**: `IsZoneCardDetected`  
- **类型**: ConditionNode  
- **功能**: 检测 RFID 是否踩到指定区域  
- **输入**: `zone` (SUPPLY/BASE/OUTPOST/CENTRAL_HIGHLAND 等), `rfid_status`(`RMUCRFIDStatus`), `robot_status`(`RMUCRobotStatus`)

## 5. 通用新建动作插件 (15个)

这些插件仅依赖黑板原始类型（int/double/bool/string），可在不同赛事系统间复用。

### 5.1 初始化

| BT 节点名 | 类型 | 功能 | 关键端口 |
|-----------|------|------|----------|
| `InitSentryConfig` | SyncAction | 输出所有静态配置到黑板 | 5 个话题名 + 13 个坐标对 + 13 个阈值 |
| `InitCmdState` | SyncAction | 初始化 cmd.state / cmd.allow_ammo_target | `cmd_state`(inout), `allow_ammo_target`(inout) |

### 5.2 决策

| BT 节点名 | 类型 | 功能 | 策略简述 |
|-----------|------|------|----------|
| `DecidePosture` | SyncAction | 选择姿态 (1攻/2防/3移) | 基地威胁→防御; 低血/高热→防御; 有目标且>50%HP→进攻 |
| `DecideEconomyCmd` | SyncAction | 决定经济行为 | 脱战+低血→远程回血; 弹低→远程补弹; 基地威胁→大能量机关 |

### 5.3 目标选择

| BT 节点名 | 类型 | 功能 |
|-----------|------|------|
| `SelectSafeRetreatGoal` | SyncAction | 从补给区/防御锚点中选最近撤退点 |
| `SelectNearestResupplyStation` | SyncAction | 从供给/基地/前哨增益区选最近补给站 |
| `SelectNearestDispelCard` | SyncAction | 选最近的祛弱卡点 |
| `SelectObjective` | SyncAction | 根据局势选战略目标 (中心高地/梯形高地/敌方堡垒等) |
| `SelectBestTarget` | SyncAction | 从雷达数据选最佳火力目标 (优先基地附近敌人) |

### 5.4 战斗

| BT 节点名 | 类型 | 功能 |
|-----------|------|------|
| `AimAtTarget` | StatefulAction | 解析 "id:x:y" 目标 → 云台瞄准 (RUNNING→SUCCESS) |
| `FireBurst` | StatefulAction | 开火 burst_ms → 暂停 pause_ms → SUCCESS |

### 5.5 驻留/等待

| BT 节点名 | 类型 | 功能 |
|-----------|------|------|
| `HoldAndHeal` | StatefulAction | 补血区驻留直到 hp ≥ hp_safe |
| `HoldForSupplyAmmoTick` | StatefulAction | 补给区等待弹量达标 |
| `HoldObjective` | StatefulAction | 目标点驻留 hold_ms，基地威胁或有敌人时提前结束 |
| `WaypointPatrol` | SyncAction | 3 点循环巡逻，到达后切换下一个 |

## 6. 通用新建条件插件 (8个)

| BT 节点名 | 功能 | SUCCESS 条件 |
|-----------|------|-------------|
| `IsCriticalState` | 危急状态 | HP < hp_critical **或** heat > heat_critical |
| `IsBaseThreatened` | 基地受威胁 | base_threat=true **或** 基地血量 < 50% |
| `HasValidTarget` | 有效目标 | has_target=true **且** best_target 非空 |
| `IsCombatAllowed` | 允许战斗 | 非虚弱 **且** 弹量>0 **且** 热量<上限 **且** HP>安全线 |
| `IsFireWindowOk` | 射击窗口 | 非虚弱 **且** 弹量>0 **且** 热量<上限 |
| `IsAmmoBelow` | 弹低 | ammo_allow < ammo_low |
| `IsAtGoal` | 到达目标 | ‖pose − goal‖ < arrive_radius |
| `IsAnyDispelCardDetected` | 有祛弱卡 | rfid_supply **或** rfid_base **或** rfid_outpost |

## 7. 共享复用插件 (来自 rmul，无需修改)

这些插件不依赖任何 RMUL/RMUC 消息类型，仅操作黑板原始类型或标准 ROS 消息。

| BT 节点名 | 功能 |
|-----------|------|
| `SendGoal` | Nav2 导航到点 (geometry_msgs::PoseStamped) |
| `CancelNavGoal` | 取消当前导航目标 |
| `MoveAround` | 在当前位置附近小范围移动 |
| `KeepRunning` | 永远返回 RUNNING（保持子树活跃） |
| `RateController` | 装饰器：限制子节点 tick 频率 |
| `IsRecoveryNeeded` | 检查恢复标志 |
| `ClearRecoveryFlag` | 清除恢复标志 |
| `InitSearchTimerIfNeeded` | 初始化搜索计时器 |

## 8. RMUC 拆分消息字段速查

> 原始 `RMUC.msg` 已拆分为 8 个独立消息，各位于 `rm_decision_interfaces/msg/RMUC/` 目录。

### 📥 RMUCGameStatus (`/game_status`, 1Hz)
```
uint8   game_progress          # 4=比赛中
uint16  stage_remain_time      # 剩余秒数
```

### 📥 RMUCRobotStatus (`/robot_status`, 10Hz)
```
uint16  current_hp / max_hp / shooter_heat / heat_limit / cooling_rate
uint16  ammo_allow / ammo_left
bool    is_dead / is_weak / is_disengaged
float32 disengage_cd_s
bool    can_remote_heal / can_remote_ammo
uint16  team_coins
bool    can_respawn
uint8   respawn_countdown_s
uint16  base_hp_cur / base_hp_max
bool    outpost_alive
```

### 📥 RMUCRFIDStatus (`/rfid_status`, 事件驱动)
```
bool    rfid_supply / rfid_base_buff / rfid_outpost_buff
bool    rfid_fortress_ally / rfid_fortress_enemy
bool    rfid_central_highland / rfid_ladder_highland
```

### 📥 RMUCRobotPosition (`/robot_position`, 50Hz)
```
float32 pose_x / pose_y / pose_yaw
bool    is_at_nav_goal
```

### 📥 RMUCEnemyTracks (`/radar/enemy_tracks`, 10-30Hz)
```
uint8     enemy_count
uint8[]   enemy_robot_id
float32[] enemy_x / enemy_y / enemy_confidence
```

### 📤 RMUCSentryCmd (`/sentry_cmd`, 2Hz)
```
uint8   cmd_posture                  # 1攻/2防/3移
bool    cmd_confirm_respawn / cmd_confirm_instant_respawn
uint16  cmd_allow_ammo_target
bool    cmd_trigger_remote_ammo / cmd_trigger_remote_hp
bool    cmd_enable_big_energy
```

### 📤 RMUCRobotControl (`/robot_control`, 10Hz)
```
bool    stop_gimbal_scan / chassis_spin / fire_enable
```

### 📤 RMUCNavControlCmd (`/nav_control_cmd`, 按需)
```
int32   cmd_type                     # 0=无操作 1=开始 2=终止 3=原地
bool    emergency_stop
```

## 9. 子树与插件引用矩阵

| 子树 | 使用的 RMUC 镜像 | 使用的新建插件 | 使用的共享插件 |
|:---|:---|:---|:---|
| PerceptionAndBlackboard | RmucSubGameStatus, RmucSubRobotStatus, RmucSubRFIDStatus, RmucSubRobotPosition | SubRadarTracks, ParseSentryBlackboard | — |
| InitOnce | — | InitSentryConfig, InitCmdState | — |
| CommandHub | — | DecidePosture, DecideEconomyCmd, DecideRespawnCmd, SentryCmdMux | KeepRunning, RateController |
| RespawnRecovery | RmucIsDead, RmucRobotControl, RmucNavControlCmd, RmucWaitAndHeal, RmucIsSupplyCardDetected, RmucIsAtNavGoal, RmucDetectRespawnAndSetRecovery, RmucMicroSearchSupplyCard | — | IsRecoveryNeeded, ClearRecoveryFlag, InitSearchTimerIfNeeded, SendGoal, KeepRunning |
| CriticalSurvival | RmucRobotControl | IsCriticalState, SelectSafeRetreatGoal | CancelNavGoal, SendGoal, KeepRunning, RateController |
| BaseDefense | RmucRobotControl | IsBaseThreatened, CombatLoop→(子树) | SendGoal, RateController |
| EngageCombat | RmucRobotControl | HasValidTarget, IsCombatAllowed, CombatLoop→(子树) | — |
| CombatLoop | RmucRobotControl | SelectBestTarget, AimAtTarget, IsFireWindowOk, FireBurst | KeepRunning |
| HealPlan | RmucIsHPBelow, RmucRobotControl | IsZoneCardDetected, HoldAndHeal | SendGoal, KeepRunning, RateController |
| AmmoPlan | RmucRobotControl | IsAmmoBelow, IsZoneCardDetected, HoldForSupplyAmmoTick, SelectNearestResupplyStation | SendGoal, KeepRunning, RateController |
| ObjectivePlanner | RmucRobotControl | SelectObjective, IsAtGoal, HoldObjective | SendGoal, KeepRunning, RateController |
| PatrolAndScan | RmucRobotControl | WaypointPatrol | SendGoal, KeepRunning, RateController |

## 10. 文件结构

```
rm_behavior_tree/
├── include/rm_behavior_tree/plugins/rmuc_2026/
│   ├── action/
│   │   ├── aim_at_target.hpp
│   │   ├── decide_economy_cmd.hpp
│   │   ├── decide_posture.hpp
│   │   ├── decide_respawn_cmd.hpp
│   │   ├── detect_respawn_and_set_recovery.hpp
│   │   ├── fire_burst.hpp
│   │   ├── hold_and_heal.hpp
│   │   ├── hold_for_supply_ammo_tick.hpp
│   │   ├── hold_objective.hpp
│   │   ├── init_cmd_state.hpp
│   │   ├── init_sentry_config.hpp
│   │   ├── micro_search_supply_card.hpp
│   │   ├── nav_control_cmd.hpp
│   │   ├── parse_sentry_blackboard.hpp
│   │   ├── robot_control.hpp
│   │   ├── select_best_target.hpp
│   │   ├── select_nearest_dispel_card.hpp
│   │   ├── select_nearest_resupply_station.hpp
│   │   ├── select_objective.hpp
│   │   ├── select_safe_retreat_goal.hpp
│   │   ├── sentry_cmd_mux.hpp
│   │   ├── sub_game_status.hpp
│   │   ├── sub_radar_tracks.hpp
│   │   ├── sub_rfid_status.hpp
│   │   ├── sub_robot_position.hpp
│   │   ├── sub_robot_status.hpp
│   │   ├── wait_and_heal.hpp
│   │   └── waypoint_patrol.hpp
│   └── condition/
│       ├── has_valid_target.hpp
│       ├── is_ammo_below.hpp
│       ├── is_any_dispel_card_detected.hpp
│       ├── is_at_goal.hpp
│       ├── is_at_nav_goal.hpp
│       ├── is_base_threatened.hpp
│       ├── is_combat_allowed.hpp
│       ├── is_critical_state.hpp
│       ├── is_dead.hpp
│       ├── is_fire_window_ok.hpp
│       ├── is_game_time.hpp
│       ├── is_hp_below.hpp
│       ├── is_supply_card_detected.hpp
│       └── is_zone_card_detected.hpp
├── plugins/rmuc_2026/  (同名 cpp 文件)
├── config/rmuc_2026/
│   ├── rmuc_2026.xml           (主树)
│   ├── PerceptionAndBlackboard.xml
│   ├── InitOnce.xml
│   ├── CommandHub.xml
│   ├── RespawnRecovery.xml
│   ├── CriticalSurvival.xml
│   ├── BaseDefense.xml
│   ├── EngageCombat.xml
│   ├── CombatLoop.xml
│   ├── SustainAndEconomy.xml
│   ├── HealPlan.xml
│   ├── AmmoPlan.xml
│   ├── ObjectivePlanner.xml
│   ├── PatrolAndScan.xml
│   └── rmuc_2026_plugins.ipynb  ← 本文档
└── CMakeLists.txt (rmuc_* 插件目标)
```

## 11. 入口 & 话题 & 构建

### 入口文件

RMUC 行为树使用独立入口 `rm_behavior_tree_rmuc.cpp`，与 RMUL 的 `rm_behavior_tree.cpp` 完全分离。

| 入口 | 可执行文件 | Groot2 端口 | 默认 XML |
|:---|:---|:---:|:---|
| RMUL | `rm_behavior_tree` | 1667 | `config/attack_left.xml` |
| **RMUC** | **`rm_behavior_tree_rmuc`** | **1668** | **`config/rmuc_2026/rmuc_2026.xml`** |

### 独立话题 (拆分后)

每个 RMUC 订阅/发布插件使用 **独立的 RosNodeParams 和话题**，不再共用 `/rmuc`：

| 方向 | BT 节点名 | 话题 | 消息类型 |
|:---:|:---|:---|:---|
| 📥 | `RmucSubGameStatus` | `/game_status` | `RMUCGameStatus` |
| 📥 | `RmucSubRobotStatus` | `/robot_status` | `RMUCRobotStatus` |
| 📥 | `RmucSubRFIDStatus` | `/rfid_status` | `RMUCRFIDStatus` |
| 📥 | `RmucSubRobotPosition` | `/robot_position` | `RMUCRobotPosition` |
| 📥 | `SubRadarTracks` | `/radar/enemy_tracks` | `RMUCEnemyTracks` |
| 📥 | `RmucDetectRespawnAndSetRecovery` | `/robot_status` | `RMUCRobotStatus` |
| 📥 | `RmucWaitAndHeal` | `/robot_status` | `RMUCRobotStatus` |
| 📤 | `SentryCmdMux` | `/sentry_cmd` | `RMUCSentryCmd` |
| 📤 | `RmucRobotControl` | `/robot_control` | `RMUCRobotControl` |
| 📤 | `RmucNavControlCmd` | `/nav_control_cmd` | `RMUCNavControlCmd` |
| 📤 | `SendGoal` | `navigate_to_pose` | PoseStamped (Nav2) |

### 构建与运行

```bash
# 编译（两个包）
cd ~/rm_code/2026_1_21/ros2_ws
colcon build --packages-select rm_decision_interfaces rm_behavior_tree

# 运行 RMUC 行为树
source install/setup.bash
ros2 run rm_behavior_tree rm_behavior_tree_rmuc \
  --ros-args -p style:=$(pwd)/src/rm_behavior_tree/rm_behavior_tree/config/rmuc_2026/rmuc_2026.xml

# 另一终端验证话题
ros2 topic list | grep -E "game_status|robot_status|rfid_status|robot_position|enemy_tracks|sentry_cmd|robot_control|nav_control"
```

## 12. 扩展指南

### 添加新插件
1. 在 `include/rm_behavior_tree/plugins/rmuc_2026/{action,condition}/` 创建 `.hpp`
2. 在 `plugins/rmuc_2026/{action,condition}/` 创建 `.cpp`
3. 在 `CMakeLists.txt` 添加 `ament_auto_add_library(rmuc_xxx SHARED ...)`
4. 如需新消息字段，在 `rm_decision_interfaces/msg/RMUC/` 目录下对应的 `.msg` 文件中添加
5. 在对应子树 XML 中引用，并在 `rmuc_2026.xml` 的 `TreeNodesModel` 中添加声明
6. 在 `rm_behavior_tree_rmuc.cpp` 中使用对应话题的 `RosNodeParams` 注册插件

### 添加新话题
若需新增独立话题（例如新的传感器输入）：
1. 在 `rm_decision_interfaces/msg/RMUC/` 下新建 `.msg`
2. 在 `rm_decision_interfaces/CMakeLists.txt` 中注册
3. 在 `rm_behavior_tree_rmuc.cpp` 中新建对应的 `BT::RosNodeParams`

### 标记说明
- 源码中带 `TODO:` 的地方表示需要根据实际硬件/赛事规则完善的逻辑
- `AimAtTarget` / `FireBurst` 需要与实际云台/射击控制器对接
- `ParseSentryBlackboard` 中的基地威胁判定需结合实际雷达数据细化